In [ ]:
import os
import sys
from glob import glob

module_path = os.path.abspath(os.path.join('../module/'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import pandas as pd
import tqdm

In [ ]:
from prostate_recurrence.image_processing import validate_psr_image

In [ ]:
# Change this to a valid directory containing psr image tiles!
directory_processed_data = '/Volumes/lab-batesp/working/Xiao/all_about_deep_learning/TME_machine_learning/processed_data/'
sub_directory_cohort = 'clinical/prostate/chiip_cohort/whole_slide'
sub_directory_sample_psr = 'C001021_B05/PSR/tile_size_1024/pre_processing/deconvolutions/psr'
file_pattern_psr = '*psr.tif'
image_paths = glob(
    os.path.join(
        directory_processed_data,
        sub_directory_cohort,
        sub_directory_sample_psr,
        file_pattern_psr
    )
)

In [ ]:
print(image_paths[:10])

In [ ]:
summary_columns = [
    'path_to_image_tile', 'row', 'column', 'valid',
    'fraction_with_min_intensity'
]

In [ ]:
summary_rows = []
for image_path in tqdm.tqdm(image_paths[:]):
    
    # read image
    image_arr = validate_psr_image.read_psr_image(image_path)
    
    # validate image
    (valid, fraction_with_min_intensity) = validate_psr_image.validate_psr_image(image_arr)
    
    # record validation summary
    summary_rows.append(
        (
            image_path,
            int(image_path.split('/')[-1].split('_')[2]),
            int(image_path.split('/')[-1].split('_')[3]),
            valid,
            fraction_with_min_intensity
        )
    )

In [ ]:
df_summary = pd.DataFrame(
    data = summary_rows,
    columns = summary_columns
)

In [ ]:
df_summary.head()

In [ ]:
df_summary.loc[df_summary.fraction_with_min_intensity >= 0.005].head(50)

In [ ]:
df_summary.shape

In [ ]:
df_summary.loc[df_summary.fraction_with_min_intensity >= 0.005].shape

In [ ]:
output_directory_summary = os.path.join(
    directory_processed_data,
    sub_directory_cohort,
    sub_directory_sample_psr,
    'validation_psr_images_summary/'
)
os.makedirs(output_directory_summary, exist_ok=True)

In [ ]:
validate_psr_image.write_validation_summary(
    summary=df_summary,
    output_directory=output_directory_summary
)

In [ ]:
output_directory_summary